# MirrorTopology Step 1 — Phase D-3b：12 位置追加配置（81）の bank 生成（family 単位；v0.1）
受入れ済み tranche 2b の依存（bank 仕様 v2・108 位置共分散 receipt・検証済み `TwelveContext`）のもとで，1 family（E2/E7/E8）の **追加 27 配置**（各 size 9；第 1 波 3 配置は D-2 の固定入力で再生成しない）の評価 bank（4×10⁶ 行×3 role，float64 のみ）と fitting bank を **D-2 と同じ family latent**（同 CRN 表・group・key）上に生成し，`d3_bank_registry.json`・call inventory・producer binding を残す。**family を 1 つずつ別 run**（さらに `SIZES` で size 1 つずつに分割可：9 配置 ≈1.5〜2.5 h の見込み；D-2 の Colab 計時からの外挿で保証ではない）。出力は Drive（≈336 MB／配置 → ≈3 GB／size，≈9 GB／family，全体 ≈27 GB；**実行前に Drive の空き容量を確認**）；repository には registry のみ。CPU runtime（High-RAM 任意；GPU 不要）。中断時は同じ commit で `REUSE_ROOT` に前回の `OUT/d3b` を指定して再開（完成 dir は context に対する全再検証＋request 一致の後にのみ再利用）。家族 reference bank（D-2）は生成も複製もしない：`D2_REF_ROOT` を与えた場合のみ ledger identity で再検証して記録。label なし・較正なし・plan 固定なし。

In [43]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '7b09c646b3691260bf06b00055c5b5ad9f8540ca'
EXPECTED_INVENTORY_SHA256 = '4767847789a01d7f74282adfd7d3e6cfc924000307fcf10abf896c29bce1be5e'
FAMILY = 'E8'
SIZES = 'L1.00'
DRIVE_DIR = '/content/drive/MyDrive/MirrorTopology_D3b'
REUSE_ROOT = ''
D2_REF_ROOT = ''
LAUNCHER_ID = 'MirrorTopology_Step1_D3b_bankgen_v0.1'

In [44]:
# --- 1. fresh scratch checkout at C; inventory bound; d/ pins+script+spec v2 bound; Phase C packet at the same commit
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E2','E7','E8'), 'launcher lock not filled'
SZ=[s.strip() for s in SIZES.split(',') if s.strip()]; assert all(s in ('L1.00','L1.20','L1.50') for s in SZ), SIZES
from google.colab import drive; drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
st=shutil.disk_usage(DRIVE_DIR); print('Drive free GB:', round(st.free/1e9,1), '(needed: ~3 GB per size, ~9 GB per family)')
RUN=f'{DRIVE_DIR}/{FAMILY}_{"all" if not SZ else "-".join(SZ)}_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'/content/d3b_scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; PHASEC=f'{MT}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/d/d3_pins.json'; SCRIPT=f'{PHASEB}/d/d3_bankgen.py'; SPEC2=f'{PHASEB}/d/d3_bank_spec.json'
assert sha(PINS)==inv['d_sha256']['d/d3_pins.json'] and sha(SCRIPT)==inv['d_sha256']['d/d3_bankgen.py'] and sha(SPEC2)==inv['d_sha256']['d/d3_bank_spec.json']
pins=json.load(open(PINS)); assert pins['schema']=='d3_pins_v1' and pins['engine_version']==inv['engine_version'] and sha(f'{PHASEC}/PACKET_INVENTORY.json')==pins['phaseC_inventory_sha256'] and json.load(open(SPEC2))['spec_sha256']==pins['bank_spec_v2_sha256']
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), spec_v2_sha256=pins['bank_spec_v2_sha256'], covariance_receipt_sha256=pins['covariance_receipt_sha256'], engine_version=inv['engine_version'], family=FAMILY, sizes=(SZ or ['L1.00','L1.20','L1.50']), run_dir=RUN, reuse_root=REUSE_ROOT or None, d2_ref_root=D2_REF_ROOT or None, drive_free_bytes=st.free); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive free GB: 191.3 (needed: ~3 GB per size, ~9 GB per family)
{'launcher_id': 'MirrorTopology_Step1_D3b_bankgen_v0.1', 'repo_url': 'https://github.com/tsujikeita/mirror-topology.git', 'commit': '7b09c646b3691260bf06b00055c5b5ad9f8540ca', 'inventory_sha256': '4767847789a01d7f74282adfd7d3e6cfc924000307fcf10abf896c29bce1be5e', 'pins_sha256': '284ca06a7115b13327be6dba458533702985fa61e3f22a3b0064b52dd1455173', 'spec_v2_sha256': 'd5fda77b4f921935b9ceb5a80df2b3c6ac41ccafa1141f877620cdd866dffcd4', 'covariance_receipt_sha256': 'ab545a6f993da5890214a54cb0b3229dc064dbf37704fb229d3095dccc98dc89', 'engine_version': '0.93.0', 'family': 'E8', 'sizes': ['L1.00'], 'run_dir': '/content/drive/MyDrive/MirrorTopology_D3b/E8_L1.00_20260930T213708Z', 'reuse_root': None, 'd2_ref_root': None, 'drive_free_bytes': 191307624448}


In [45]:
# --- 2. registered environment (the script re-verifies live versions and pools as a HARD gate); no CMBtopology needed
ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
print('environment ok', live)


environment ok {'numpy': '2.1.3', 'scipy': '1.16.3', 'healpy': '1.20.0', 'camb': '2.0.4', 'pot': '0.9.7.post1', 'python': '3.13.15'}


In [46]:
# --- 3. generation (long); output goes straight to Drive
DO=f'{OUT}/d3b'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',DO,'--family',FAMILY,'--profile','production_official']+(['--sizes',','.join(SZ)] if SZ else [])+(['--reuse-root',REUSE_ROOT] if REUSE_ROOT else [])+(['--d2-ref-root',D2_REF_ROOT] if D2_REF_ROOT else [])
rc=subprocess.run(args,capture_output=True,text=True); open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-3000:])
try:
    rm=json.load(open(f'{DO}/d3_run_manifest.json'))
    if not isinstance(rm, dict) or not isinstance(rm.get('D3B_PASS'), bool):
        raise ValueError('run manifest is not a result dictionary with boolean D3B_PASS')
except Exception as read_error:
    rm=dict(D3B_PASS=False, stage='RECORD_MISSING_OR_INVALID', gates={}, directories={}, scope={}, launcher_fallback=True,
            failures=[f'run manifest unreadable: {read_error!r}', f'script returncode {rc.returncode}'], stderr_tail=rc.stderr[-2000:])
script_ok=bool(rc.returncode==0 and rm.get('D3B_PASS') is True and not rm.get('launcher_fallback'))
print('script rc', rc.returncode, 'D3B_PASS', rm.get('D3B_PASS'), rm.get('failures'))


  generated cfg40104_b0 (1000000 rows, 205s)
  generated cfg40104_b1 (3000000 rows, 623s)
  generated cfg40104_fit (200000 rows, 43s)
config 40104 (L1.00 position 3) done (872s)
  generated cfg40105_b0 (1000000 rows, 203s)
  generated cfg40105_b1 (3000000 rows, 618s)
  generated cfg40105_fit (200000 rows, 43s)
config 40105 (L1.00 position 4) done (866s)
  generated cfg40106_b0 (1000000 rows, 211s)
  generated cfg40106_b1 (3000000 rows, 612s)
  generated cfg40106_fit (200000 rows, 49s)
config 40106 (L1.00 position 5) done (872s)
  generated cfg40107_b0 (1000000 rows, 205s)
  generated cfg40107_b1 (3000000 rows, 618s)
  generated cfg40107_fit (200000 rows, 43s)
config 40107 (L1.00 position 6) done (867s)
  generated cfg40108_b0 (1000000 rows, 203s)
  generated cfg40108_b1 (3000000 rows, 617s)
  generated cfg40108_fit (200000 rows, 44s)
config 40108 (L1.00 position 7) done (864s)
  generated cfg40109_b0 (1000000 rows, 201s)
  generated cfg40109_b1 (3000000 rows, 608s)
  generated cfg40109

In [47]:
# --- 4. final record (registry + manifests + inventory of the Drive output; the bank arrays stay on Drive) + small zip for the audit
def inventory(root, exclude=(), skip_ext=('.npz',)):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue
            out[rel]=dict(sha256=(sha(p) if not f.endswith(skip_ext) else None), bytes=os.path.getsize(p))
    return out
final=dict(launcher=lock, D3B_PASS=bool(script_ok), family=FAMILY, sizes=lock['sizes'], stages=dict(checkout=True, script_returncode=rc.returncode, script_pass=rm.get('D3B_PASS'), stage=rm.get('stage'), launcher_fallback=bool(rm.get('launcher_fallback')),  script_gates=rm.get('gates'), script_failures=rm.get('failures'), scope=rm.get('scope'), directories=rm.get('directories'), timings=rm.get('timings'), seconds=rm.get('seconds')))
final['output_inventory']=inventory(OUT, exclude=('d3b_final_record.json',)); json.dump(final, open(f'{OUT}/d3b_final_record.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('D3B_PASS','family','sizes')}, indent=1), 'run dir:', RUN)
import zipfile
zp=f'/content/d3b_{FAMILY}_{"all" if not SZ else "-".join(SZ)}_{REPO_COMMIT[:12]}_audit.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for d,_,fs in os.walk(OUT):
        for f in fs:
            if not f.endswith('.npz'): z.write(os.path.join(d,f), os.path.relpath(os.path.join(d,f), OUT))
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)


{
 "D3B_PASS": true,
 "family": "E8",
 "sizes": [
  "L1.00"
 ]
} run dir: /content/drive/MyDrive/MirrorTopology_D3b/E8_L1.00_20260930T213708Z
/content/d3b_E8_L1.00_7b09c646b369_audit.zip 290213


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>